# L1 · LeRobot de un vistazo

## Por qué existe

Hasta hace poco, cada paper de robot learning venía con lo suyo: su formato
de datos, su código de entrenamiento, sus scripts de evaluación, su forma de
hablarle al robot. Querer comparar ACT contra Diffusion Policy sobre *tus*
datos era reescribir media cosa. Y compartir un dataset era mandar una carpeta
con un README rezando que el otro entendiera el formato.

**LeRobot** (de Hugging Face) junta todo eso en una librería:

- **Un formato de dataset común**, que cualquier política entiende.
- **Políticas con la misma interfaz**: cambiar ACT por Diffusion es cambiar una
  línea de configuración.
- **Herramientas** para grabar, mirar, editar, entrenar y evaluar.
- **El Hub**: datasets y modelos compartidos como se comparten modelos de
  lenguaje.

Este notebook es el mapa. Los demás de `learn/` entran en cada pieza.

- **Lo principal (1–4):** las piezas, el ciclo de trabajo, las políticas, y
  cómo se le pasa configuración.
- **Para profundizar (5–6):** datasets públicos del Hub e inferencia remota.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import importlib
import pkgutil

import lerobot
import pandas as pd

print("LeRobot", lerobot.__version__)

## 1. Las piezas

Las carpetas del paquete, y si las usamos:

| Pieza | Qué es | ¿La usamos? |
|---|---|---|
| `datasets` | El formato y la clase `LeRobotDataset` para leerlo | Sí, todo el tiempo |
| `policies` | Las políticas (ACT, Diffusion, VLAs...) | Sí: ACT hoy, otras después |
| `processor` | Normalización y conversiones antes y después de la red | Sí (notebook L4) |
| `configs` | Las configuraciones como *dataclasses* de Python | Sí, indirectamente |
| `scripts` | Los comandos `lerobot-*` (entrenar, evaluar, grabar...) | Entrenar y editar datasets |
| `envs` | Simuladores para benchmarks (PushT, ALOHA, LIBERO, MetaWorld) | Para reproducir papers |
| `robots`, `teleoperators`, `cameras`, `motors` | Hablarle a hardware real | **No**: el G1 no está soportado |
| `async_inference` | Correr la política en otra máquina | Posiblemente (sección 6) |
| `rl` | Aprendizaje por refuerzo con intervención humana (HIL-SERL) | No por ahora |

Lo de hardware es importante: LeRobot trae brazos baratos (SO-100/101, Koch),
el LeKiwi y otros, pero **no el G1**. Por eso Unitree tiene su propio
`unitree_lerobot` y su `xr_teleoperate`: graban en su formato, y nosotros
convertimos a LeRobot (notebook 02).

Lo que hay registrado en esta instalación:

In [ ]:
import lerobot.policies, lerobot.robots, lerobot.teleoperators
from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import EnvConfig
from lerobot.robots.config import RobotConfig
from lerobot.teleoperators.config import TeleoperatorConfig

# Cada pieza se registra al importar su módulo; importamos todo para ver el catálogo completo.
for package in (lerobot.policies, lerobot.robots, lerobot.teleoperators):
    for module in pkgutil.iter_modules(package.__path__):
        try:
            importlib.import_module(f"{package.__name__}.{module.name}")
        except ImportError:
            pass  # algunas piezas necesitan dependencias opcionales que no instalamos

for label, registry in [("políticas", PreTrainedConfig), ("simuladores", EnvConfig),
                        ("robots", RobotConfig), ("teleoperadores", TeleoperatorConfig)]:
    print(f"{label:15s}", sorted(registry.get_known_choices()))

## 2. El ciclo de trabajo

Con un robot soportado, el ciclo completo con LeRobot es:

```
teleoperar → grabar → mirar/editar → entrenar → evaluar (sim) → evaluar (robot)
```

Y así lo cubrimos con el G1:

| Paso | Comando de LeRobot | Con el G1 |
|---|---|---|
| Teleoperar | `lerobot-teleoperate` | `xr_teleoperate` de Unitree (visor XR) |
| Grabar | `lerobot-record` | `xr_teleoperate` graba JSON + JPEG |
| Convertir | (no hace falta) | `toolkit.dataset.convert_unitree` (notebook 02) |
| Mirar | `lerobot-dataset-viz` | notebooks 01 y 04 |
| Editar | `lerobot-edit-dataset` | el mismo |
| Entrenar | `lerobot-train` | el mismo (notebook L5) |
| Evaluar offline | (no trae) | `toolkit.eval.offline` (notebook 05) |
| Evaluar en sim | `lerobot-eval` | pendiente: escena del G1 en Isaac |
| Evaluar en el robot | `lerobot-record --policy...` | `eval_g1.py` de `unitree_lerobot` |

Los comandos están instalados en el entorno; todos aceptan `--help`.

In [ ]:
import subprocess, sys
from pathlib import Path

bin_dir = Path(sys.executable).parent
print("\n".join(sorted(p.name for p in bin_dir.glob("lerobot-*"))))

## 3. Las políticas: misma interfaz, distintas ideas

| Política | Idea en una línea | Tipo |
|---|---|---|
| `act` | Bloques de acciones con un transformer (notebook L3) | Imitación, chica |
| `diffusion` | Genera el bloque "limpiando ruido", como los generadores de imágenes; maneja bien varios estilos de demostración | Imitación |
| `vqbet` | Discretiza las acciones en un vocabulario y las predice como tokens | Imitación |
| `smolvla` | Modelo visión-lenguaje-acción chico de Hugging Face: recibe la tarea en texto | VLA |
| `pi0`, `pi05` | Los VLA de Physical Intelligence | VLA, grande |
| `groot` | GR00T N1.5 de NVIDIA, pensado para humanoides | VLA, grande |
| `tdmpc`, `sac`, `reward_classifier` | Aprendizaje por refuerzo | RL |

Todas comparten la misma interfaz, que es lo que permite cambiar de modelo sin
cambiar el resto del código:

- `forward(lote)` → pérdida, para entrenar.
- `predict_action_chunk(observación)` → un bloque de acciones.
- `select_action(observación)` → una acción, manejando el bloque por adentro.
- `reset()` → olvidar el estado entre episodios.

### Lo que cambia: qué datos pide cada una

Acá está lo más importante para entender "cómo espera los datos" una política.
Cada configuración declara **qué ventana de tiempo** necesita de cada clave, y
LeRobot arma esa ventana desde el dataset (es el `delta_timestamps` del
notebook 02). Mirá ACT contra Diffusion sobre **nuestro** dataset:

In [ ]:
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from toolkit.paths import DATASETS_DIR

meta = LeRobotDatasetMetadata("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1")
fps = meta.fps

def describe(window):
    frames = [round(t * fps) for t in window]
    return f"{len(frames)} cuadros, del {frames[0]:+d} al {frames[-1]:+d}"

for config in [ACTConfig(chunk_size=48, n_action_steps=48), DiffusionConfig()]:
    print(type(config).__name__.replace("Config", ""))
    windows = resolve_delta_timestamps(config, meta) or {}
    for key in ["observation.state", "observation.images.cam_right_high", "action"]:
        print(f"   {key:36s}", describe(windows[key]) if key in windows else "sólo el cuadro actual")

- **ACT** mira **sólo el presente** y predice **48 acciones** hacia adelante.
- **Diffusion** mira **2 cuadros** (el actual y el anterior: así puede "ver"
  velocidad) y predice **16 acciones**, empezando una antes del presente; de
  esas ejecuta 8.

Mismo dataset, distintas preguntas. Por eso el dataset guarda todo cuadro por
cuadro y la ventana la arma la política: el formato no se casa con ningún
modelo.

Y el tamaño. Instanciamos un Diffusion con los defaults, sin entrenar, sólo
para contar parámetros:

In [ ]:
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.configs.types import FeatureType
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

features = dataset_to_policy_features(meta.features)
outputs = {k: v for k, v in features.items() if v.type is FeatureType.ACTION}
inputs = {k: v for k, v in features.items() if k not in outputs}

act_default = ACTPolicy(ACTConfig(input_features=inputs, output_features=outputs))
diffusion = DiffusionPolicy(DiffusionConfig(input_features=inputs, output_features=outputs, crop_shape=(320, 576)))
count = lambda m: f"{sum(p.numel() for p in m.parameters()) / 1e6:.0f} M"
print("ACT (defaults de LeRobot):", count(act_default))
print("Diffusion (defaults)     :", count(diffusion))
print("nuestro ACT liviano      : ~15 M (notebook L3)")

Pasar de ACT a Diffusion es cambiar la configuración... pero es un modelo más
de 5 veces más grande y con otra forma de pedir datos. "Mismo código" no
significa "mismo costo".

## 4. Cómo se configura todo

Cada pieza tiene una configuración que es una *dataclass* de Python. Desde la
terminal, cada campo se vuelve una opción con puntos:

```
lerobot-train --policy.type=act --policy.chunk_size=48 --batch_size=64 --steps=30000 ...
```

`--policy.chunk_size=48` pone `chunk_size = 48` adentro de la configuración de
la política. La librería que hace esa traducción se llama *draccus*.

Nuestro checkpoint guarda la configuración completa con que se entrenó
(`train_config.json`). Reconstruimos el comando equivalente (salteando rutas
locales y datos internos):

In [ ]:
import json
from toolkit.paths import CHECKPOINTS_DIR

train = json.loads((CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model" / "train_config.json").read_text())
policy_keys = ["type", "chunk_size", "n_action_steps", "dim_model", "dim_feedforward", "n_heads",
               "kl_weight", "optimizer_lr", "optimizer_lr_backbone"]
flags = [f"--policy.{k}={train['policy'][k]}" for k in policy_keys]
flags += [f"--{k}={train[k]}" for k in ["steps", "batch_size", "seed"]]
flags += [f"--dataset.image_transforms.enable={train['dataset']['image_transforms']['enable']}"]
print("lerobot-train \\\n    " + " \\\n    ".join(["--dataset.repo_id=local/pick_mameluco_act_v1"] + flags))

Fijate `image_transforms.enable=False`: se entrenó **sin aumentación de
imágenes** (variar un poco brillo, contraste, recorte en cada lote para que la
red no memorice píxeles). Con 54 episodios y el sobreajuste que vimos en el
notebook 05, es una de las primeras perillas a probar.

---

# Para profundizar

## 5. El Hub: datasets y modelos públicos

Hay cientos de datasets en formato LeRobot en
[huggingface.co/lerobot](https://huggingface.co/lerobot). Se pueden leer igual
que el nuestro. Bajamos sólo la metadata (unos KB) de uno clásico: el de ALOHA
simulado del paper de ACT.

In [ ]:
aloha = LeRobotDatasetMetadata("lerobot/aloha_sim_transfer_cube_human")
ours = meta
pd.DataFrame({
    "ALOHA sim (paper de ACT)": {"episodios": aloha.total_episodes, "cuadros": aloha.total_frames, "fps": aloha.fps,
                                  "cámaras": len(aloha.camera_keys), "dim. acción": aloha.features["action"]["shape"][0]},
    "nuestro G1": {"episodios": ours.total_episodes, "cuadros": ours.total_frames, "fps": ours.fps,
                   "cámaras": len(ours.camera_keys), "dim. acción": ours.features["action"]["shape"][0]},
})

El paper de ACT resolvió esta tarea simulada con 50 demostraciones de 400
cuadros; nosotros tenemos 61 de una tarea real, con 26 dimensiones en vez de
14. Para el camino de aprendizaje, la idea es reproducir ACT primero sobre este
dataset (donde sabemos qué número tiene que salir) y recién después
preocuparnos por el nuestro.

Para subir o bajar un dataset completo: `LeRobotDataset("usuario/nombre")` lo
baja; `dataset.push_to_hub()` lo sube. Los modelos igual:
`ACTPolicy.from_pretrained("lerobot/act_aloha_sim_transfer_cube_human")`.

**Ojo con el nuestro:** las imágenes muestran el lugar real; si alguna vez se
sube, tiene que ser **privado**.

## 6. Inferencia remota (`async_inference`)

En el Jetson del robot, ACT tarda 312 ms por bloque en CPU. Para modelos más
grandes (Diffusion, VLAs) eso no alcanza. LeRobot trae un esquema
cliente-servidor: la política corre en una máquina con GPU (`policy_server`) y
el robot (`robot_client`) le manda observaciones y recibe bloques por la red,
**mientras sigue ejecutando el bloque anterior**. Así la latencia de red y de
cómputo queda escondida detrás de la ejecución.

Está pensado para los robots que LeRobot soporta; con el G1 habría que adaptar
el cliente. Lo anotamos para cuando lleguemos a los VLAs.

## Referencias

- Documentación: [huggingface.co/docs/lerobot](https://huggingface.co/docs/lerobot)
- Código: [github.com/huggingface/lerobot](https://github.com/huggingface/lerobot)
- Datasets y modelos: [huggingface.co/lerobot](https://huggingface.co/lerobot)